# 잇다 모델 A 학습: EXAONE 4.0 1.2B (LoRA, bf16) - 집 PC용
RTX 4070 Ti 12GB, WSL. Gemma 계열은 다른 팀원이 진행하므로 EXAONE로 비교 실험.

- 모델: `LGAI-EXAONE/EXAONE-4.0-1.2B` (transformers 내장 구조라 원격 코드 없음, 한국어 지원)
- 방식: Unsloth 없이 transformers + PEFT(LoRA). 1.2B라 4비트로 줄이지 않고 bf16 그대로 학습 (약 2.5GB)
- 추론은 생각 모드 없이(non-reasoning) 사용
- 라이선스: EXAONE AI Model License 1.2 - NC (비상업). 교육 프로젝트 사용은 가능, 상용화 시 별도 협의 필요

**반드시 새 커널에서 실행** (Unsloth를 불러온 커널이면 transformers가 바뀌어 있을 수 있음). VS Code에서 커널 재시작 후 위에서부터.

## 1. 경로와 설정

In [1]:
import os, json, time, re
from pathlib import Path
from collections import Counter
import torch
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

ROOT = Path.cwd()
while not (ROOT / "ml" / "data" / "train.jsonl").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT / "ml" / "data"
OUT  = ROOT / "ml" / "train" / "out_exaone"
OUT.mkdir(parents = True, exist_ok = True)
MODEL_ID = "LGAI-EXAONE/EXAONE-4.0-1.2B"
MAX_LEN  = 1024
SYSTEM   = (ROOT / "config" / "system_prompt.txt").read_text(encoding = "utf-8").strip()
print("레포:", ROOT, "| GPU:", torch.cuda.get_device_name(0), "| bf16:", torch.cuda.is_bf16_supported())
import transformers, peft; print("transformers", transformers.__version__, "| peft", peft.__version__)   # transformers 4.54 이상이어야 함

/home/aiuser/projects/itda-ai/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


레포: /home/aiuser/projects/itda-ai | GPU: NVIDIA GeForce RTX 4070 Ti | bf16: True


W0928 22:35:19.101000 24741 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0928 22:35:19.134000 24741 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


transformers 5.5.0 | peft 0.21.0


## 2. 모델과 토크나이저

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype = torch.bfloat16, device_map = "cuda")
print("eos:", tok.eos_token, "| pad:", tok.pad_token, "| GPU", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

def prompt_of(memo):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": memo}]
    return tok.apply_chat_template(msgs, add_generation_prompt = True, tokenize = False, enable_thinking = False)

print(prompt_of("어젯밤 두 번 깨심")[-300:])   # 끝이 assistant 차례로 끝나야 함

Loading weights: 100%|██████████| 332/332 [00:00<00:00, 825.07it/s]


eos: [|endofturn|] | pad: [PAD] | GPU 2.38 GB
 의문형("부스럭거리셨나?")은 출력하지 않는다. "~것 같음", "아마" 같은 추측은 present로 출력한다
- 반복 질문, 실금, 낮잠처럼 목록에 없는 관찰은 가까운 유형으로 넣지 않고 출력하지 않는다
- 보호자·가족의 상태, 의료진의 말, 오래된 과거 회상, 특정 시간대의 좋은 모습("오후엔 기분 좋으셨음")은 출력하지 않는다
- 날짜를 계산하지 않는다. 설명 없이 JSON만 출력한다[|endofturn|]
[|user|]
어젯밤 두 번 깨심[|endofturn|]
[|assistant|]
<think>

</think>




## 3. 평가 함수 (기획안 8-4 ①)
같은 메모에서 유형(type)과 있었음/없었음(status)이 같으면 맞은 사건. JSON 통과율, 빈 결과 정확도, time_expr 일치율도 함께.

In [3]:
def parse_events(text):
    t = text.strip().replace("```json", "").replace("```", "")
    t = re.sub(r"<think>.*?</think>", "", t, flags = re.S)
    a, b = t.find("{"), t.rfind("}")
    if a < 0 or b < 0: return None
    try:
        obj = json.loads(t[a:b + 1]); ev = obj.get("events") if isinstance(obj, dict) else None
        return ev if isinstance(ev, list) else None
    except Exception:
        return None

@torch.no_grad()
def generate(memo, max_new_tokens = 384):
    enc = tok(prompt_of(memo), return_tensors = "pt", add_special_tokens = False).to("cuda")
    out = model.generate(**enc, max_new_tokens = max_new_tokens, do_sample = False, pad_token_id = tok.pad_token_id)
    return tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens = True)

def evaluate(file, tag, n = None, show = 3):
    model.eval()
    rows = [json.loads(l) for l in open(DATA / file, encoding = "utf-8") if l.strip()][:n]
    tp = fp = fn = 0; json_ok = 0; e_tot = e_ok = 0; te_m = te_t = 0; per = {}; recs = []; t0 = time.time()
    for i, r in enumerate(rows):
        if "messages" in r: memo = r["messages"][1]["content"]; gold = json.loads(r["messages"][2]["content"])["events"]
        else: memo = r["memo"]; gold = r["gold"]["events"]
        raw = generate(memo); pred = parse_events(raw)
        if pred is not None: json_ok += 1
        pred = [e for e in (pred or []) if isinstance(e, dict)]
        g = Counter((e["type"], e["status"]) for e in gold); p = Counter((e.get("type"), e.get("status")) for e in pred); hit = g & p
        tp += sum(hit.values()); fp += sum((p - g).values()); fn += sum((g - p).values())
        for (t, _), c in g.items(): per.setdefault(t, [0, 0, 0]); per[t][2] += c
        for (t, _), c in p.items(): per.setdefault(t, [0, 0, 0]); per[t][1] += c
        for (t, _), c in hit.items(): per[t][0] += c
        if not gold: e_tot += 1; e_ok += int(len(pred) == 0)
        pd = {(e.get("type"), e.get("status")): e.get("time_expr") for e in pred}
        for e in gold:
            k = (e["type"], e["status"])
            if k in pd: te_t += 1; te_m += int(pd[k] == e["time_expr"])
        recs.append({"memo": memo, "gold": gold, "pred_raw": raw})
        if i < show: print(f"[{i}] 메모: {memo}\n    예측: {raw.strip()[:300]}\n    정답: {json.dumps(gold, ensure_ascii = False)[:300]}\n")
        if (i + 1) % 50 == 0: print(f"  {i + 1}/{len(rows)}건, {round(time.time() - t0)}초")
    P = tp / (tp + fp) if tp + fp else 0; R = tp / (tp + fn) if tp + fn else 0; F = 2 * P * R / (P + R) if P + R else 0
    res = {"tag": tag, "model": MODEL_ID, "file": file, "n": len(rows), "precision": round(P, 4), "recall": round(R, 4), "f1": round(F, 4),
           "json_ok_rate": round(json_ok / len(rows), 4), "empty_correct_rate": round(e_ok / e_tot, 4) if e_tot else None,
           "time_expr_match_rate": round(te_m / te_t, 4) if te_t else None,
           "per_type_f1": {t: round(2 * v[0] / (v[1] + v[2]), 3) if v[1] + v[2] else None for t, v in sorted(per.items())},
           "sec_per_memo": round((time.time() - t0) / len(rows), 2)}
    json.dump({"summary": res, "records": recs}, open(OUT / f"eval_{tag}.json", "w", encoding = "utf-8"), ensure_ascii = False, indent = 1)
    print(json.dumps(res, ensure_ascii = False, indent = 1)); return res

## 4. 파인튜닝 전 평가 (synth_eval 300)

In [4]:
before = evaluate("synth_eval.jsonl", tag = "before_test")

[0] 메모: 우리 양반이 며칠 전에는 문 열고 나갈라 카는 거 없었다. 근데 별것도 아인 걸로 성을 세 번이나 내데. 약 묵으라 카니 안 묵는다꼬 손으로 밀어내삤다 아이가.
    예측: ```json
[
  {
    "type": "irritability",
    "status": "present",
    "time_expr": "며칠 전",
    "count": 1,
    "evidence": "성을 세 번이나 내데"
  },
  {
    "type": "agitation",
    "status": "present",
    "time_expr": "아이가",
    "count": 1,
    "evidence": "손으로 밀어내삤다"
  },
  {
    "type": "medication_re
    정답: [{"type": "wandering_exit", "status": "absent", "time_expr": "며칠 전", "count": 1, "evidence": "며칠 전에는 문 열고 나갈라 카는 거 없었다"}, {"type": "irritability", "status": "present", "time_expr": null, "count": 3, "evidence": "성을 세 번이나 내데"}, {"type": "medication_refusal", "status": "present", "time_expr": null, "c

[1] 메모: 할머니 나를 엄마로 두 번 부르심. 약 드시라니까 싫다고 안 드심. 어제 저녁엔 주무시다가 두 번 깨심
    예측: ```json
[
  {
    "type": "night_waking",
    "status": "present",
    "time_expr": "어젯밤",
    "count": 1,
    "evidence": "어제 저녁엔 plasmon하심"
  },
  {
    "type": "medication_refusal",
    "status": "present",

## 5. 학습 데이터
프롬프트(시스템+메모) 부분은 정답에서 빼고(-100), JSON 답과 끝 토큰만 학습.

In [5]:
from datasets import load_dataset
def build(ex):
    memo = ex["messages"][1]["content"]; answer = ex["messages"][2]["content"] + tok.eos_token
    p_ids = tok(prompt_of(memo), add_special_tokens = False)["input_ids"]
    a_ids = tok(answer, add_special_tokens = False)["input_ids"]
    ids = (p_ids + a_ids)[:MAX_LEN]; labels = ([-100] * len(p_ids) + a_ids)[:MAX_LEN]
    return {"input_ids": ids, "attention_mask": [1] * len(ids), "labels": labels}

train_ds = load_dataset("json", data_files = str(DATA / "train.jsonl"), split = "train")
val_ds   = load_dataset("json", data_files = str(DATA / "val.jsonl"),   split = "train")
train_ds = train_ds.map(build, remove_columns = train_ds.column_names)
val_ds   = val_ds.map(build,   remove_columns = val_ds.column_names)
lens = [len(x) for x in train_ds["input_ids"]]
print(len(train_ds), len(val_ds), "| 토큰 최대", max(lens), "평균", sum(lens) // len(lens), "| 1024 초과", sum(l >= MAX_LEN for l in lens))
ex = train_ds[0]
print("[학습 대상]", tok.decode([t for t, l in zip(ex["input_ids"], ex["labels"]) if l != -100]))   # JSON 답만 보여야 함

Map: 100%|██████████| 200/200 [00:00<00:00, 925.45 examples/s] 

1977 200 | 토큰 최대 978 평균 791 | 1024 초과 0
[학습 대상] {"events": [{"type": "irritability", "status": "present", "time_expr": null, "count": 1, "evidence": "괜히 짜증 내심"}, {"type": "wandering_exit", "status": "present", "time_expr": "2일 전", "count": 2, "evidence": "2일 전에는 현관문 열고 나가시려는 걸 두 번 말림"}, {"type": "night_waking", "status": "present", "time_expr": "아까", "count": 3, "evidence": "아까 주무시다가 세 번이나 깨심"}]}[|endofturn|]


## 6. LoRA (r 16, alpha 32)

In [6]:
from peft import LoraConfig, get_peft_model
model.gradient_checkpointing_enable(); model.enable_input_require_grads(); model.config.use_cache = False
model = get_peft_model(model, LoraConfig(
    r = 16, lora_alpha = 32, lora_dropout = 0.05, bias = "none", task_type = "CAUSAL_LM",
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
model.print_trainable_parameters()

trainable params: 15,237,120 || all params: 1,294,628,608 || trainable%: 1.1769


## 7. 학습
1977건 × 2에폭 ÷ 실질 배치 8 = 약 494스텝. 끊기면 1, 2, 5, 6을 다시 실행하고 `trainer.train(resume_from_checkpoint = True)`.

In [7]:
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq
args = TrainingArguments(
    output_dir = str(OUT / "checkpoints"),
    per_device_train_batch_size = 4, per_device_eval_batch_size = 4, gradient_accumulation_steps = 2,
    num_train_epochs = 2, learning_rate = 2e-4, warmup_steps = 10, lr_scheduler_type = "linear", weight_decay = 0.0,
    bf16 = True, logging_steps = 10,
    eval_strategy = "steps", eval_steps = 100, save_strategy = "steps", save_steps = 100, save_total_limit = 2,
    report_to = "none", seed = 3407, remove_unused_columns = False,
)
trainer = Trainer(model = model, args = args, train_dataset = train_ds, eval_dataset = val_ds,
                  data_collator = DataCollatorForSeq2Seq(tok, padding = True, label_pad_token_id = -100))
t0 = time.time()
trainer.train()
print(f"학습 {round((time.time() - t0) / 60, 1)}분, 최대 VRAM {round(torch.cuda.max_memory_reserved() / 1024**3, 2)}GB")
model.save_pretrained(str(OUT / "lora_itda")); tok.save_pretrained(str(OUT / "lora_itda"))

{'loss': '0.2621', 'grad_norm': '0.4336', 'learning_rate': '0.00018', 'epoch': '0.0404'}
{'loss': '0.1236', 'grad_norm': '0.2946', 'learning_rate': '0.0001963', 'epoch': '0.08081'}
{'loss': '0.08717', 'grad_norm': '0.2278', 'learning_rate': '0.0001922', 'epoch': '0.1212'}
{'loss': '0.05768', 'grad_norm': '0.2466', 'learning_rate': '0.0001881', 'epoch': '0.1616'}
{'loss': '0.05763', 'grad_norm': '0.263', 'learning_rate': '0.000184', 'epoch': '0.202'}
{'loss': '0.05515', 'grad_norm': '0.2439', 'learning_rate': '0.0001798', 'epoch': '0.2424'}
{'loss': '0.05962', 'grad_norm': '0.2165', 'learning_rate': '0.0001757', 'epoch': '0.2828'}
{'loss': '0.04891', 'grad_norm': '0.2588', 'learning_rate': '0.0001716', 'epoch': '0.3232'}
{'loss': '0.02799', 'grad_norm': '0.2095', 'learning_rate': '0.0001675', 'epoch': '0.3636'}
{'loss': '0.04013', 'grad_norm': '0.2624', 'learning_rate': '0.0001634', 'epoch': '0.404'}
{'eval_loss': '0.04855', 'eval_runtime': '52.94', 'eval_samples_per_second': '3.778', '

('/home/aiuser/projects/itda-ai/ml/train/out_exaone/lora_itda/tokenizer_config.json',
 '/home/aiuser/projects/itda-ai/ml/train/out_exaone/lora_itda/chat_template.jinja',
 '/home/aiuser/projects/itda-ai/ml/train/out_exaone/lora_itda/tokenizer.json')

## 8. 파인튜닝 후 평가와 비교

In [8]:
model.config.use_cache = True
after = evaluate("synth_eval.jsonl", tag = "after_test")
for k in ["precision", "recall", "f1", "json_ok_rate", "empty_correct_rate", "time_expr_match_rate", "sec_per_memo"]:
    print(f"{k:22}{str(before[k]):>10} → {after[k]}")
for t in sorted(set(before["per_type_f1"]) | set(after["per_type_f1"])):
    print(f"  {t:20} {before['per_type_f1'].get(t)} → {after['per_type_f1'].get(t)}")

[0] 메모: 우리 양반이 며칠 전에는 문 열고 나갈라 카는 거 없었다. 근데 별것도 아인 걸로 성을 세 번이나 내데. 약 묵으라 카니 안 묵는다꼬 손으로 밀어내삤다 아이가.
    예측: {"events": [{"type": "wandering_exit", "status": "absent", "time_expr": "며칠 전", "count": 1, "evidence": "며칠 전에는 문 열고 나갈라 카는 거 없었다"}, {"type": "delusion", "status": "present", "time_expr": null, "count": 3, "evidence": "별것도 아인 으로 성을 세 번이나 내데"}, {"type": "medication_refusal", "status": "present", "tim
    정답: [{"type": "wandering_exit", "status": "absent", "time_expr": "며칠 전", "count": 1, "evidence": "며칠 전에는 문 열고 나갈라 카는 거 없었다"}, {"type": "irritability", "status": "present", "time_expr": null, "count": 3, "evidence": "성을 세 번이나 내데"}, {"type": "medication_refusal", "status": "present", "time_expr": null, "c

[1] 메모: 할머니 나를 엄마로 두 번 부르심. 약 드시라니까 싫다고 안 드심. 어제 저녁엔 주무시다가 두 번 깨심
    예측: {"events": [{"type": "confusion", "status": "present", "time_expr": null, "count": 2, "evidence": "나를 엄마로 두 번 부르심"}, {"type": "medication_refusal", "status": "present", "time_expr": null, "count": 1, "evidence

## 9. LoRA 합치기 → GGUF(Q8_0)
1.2B라 Q8_0도 약 1.4GB로 작아서 Q4_K_M으로 줄이지 않고 그대로 배포해도 됨. llama.cpp의 변환 스크립트만 쓰고 빌드는 하지 않음.

In [9]:
merged = model.merge_and_unload()
merged.save_pretrained(str(OUT / "merged"), safe_serialization = True); tok.save_pretrained(str(OUT / "merged"))
print(os.listdir(OUT / "merged"))

Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.58s/it]

['chat_template.jinja', 'model.safetensors', 'tokenizer_config.json', 'generation_config.json', 'tokenizer.json', 'config.json']


In [10]:
import sys, subprocess
LC = OUT / "llama.cpp"
if not LC.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp", str(LC)], check = True)
subprocess.run(["uv", "pip", "install", "--python", sys.executable, str(LC / "gguf-py")], check = True)   # gguf만 설치 (다른 패키지 버전은 건드리지 않음)
subprocess.run([sys.executable, str(LC / "convert_hf_to_gguf.py"), str(OUT / "merged"),
                "--outfile", str(OUT / "itda-exaone-q8_0.gguf"), "--outtype", "q8_0"], check = True)
print(round((OUT / "itda-exaone-q8_0.gguf").stat().st_size / 1024**3, 2), "GB")

Cloning into '/home/aiuser/projects/itda-ai/ml/train/out_exaone/llama.cpp'...
Using Python 3.12.3 environment at: /home/aiuser/projects/itda-ai/.venv
Resolved 9 packages in 68ms
   Building gguf @ file:///home/aiuser/projects/itda-ai/ml/train/out_exaone/llama.cpp/gguf-py
      Built gguf @ file:///home/aiuser/projects/itda-ai/ml/train/out_exaone/llama.cpp/gguf-py
Prepared 1 package in 450ms
Installed 1 package in 1ms
 + gguf==0.19.0 (from file:///home/aiuser/projects/itda-ai/ml/train/out_exaone/llama.cpp/gguf-py)
INFO:hf-to-gguf:Loading model: merged
INFO:hf-to-gguf:Model architecture: Exaone4ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:rope_freqs.weight,                    torch.float32 --> F32, shape = {32}
INFO:hf-to-gguf:token_embd.weight,                    torch.bfloat16 --> Q8_0, shape = {2048, 102400}
INFO:hf-to-gguf:blk.0.ffn_down

1.27 GB
